In [1]:

"""
Fill lifetime + fleet vintage distribution parameters in an Excel sheet by querying OpenAI.

Interpretation (per your specification):
- year 0 = "now"
- distribution describes *vintage* (commissioning/installation year) relative to year 0, so values are typically <= 0.
  Example: uniform between -50 and -5 => type=4, minimum=-50, maximum=-5

Columns filled:
- lifetime (technical lifetime, years; > 0)
- age distribution type (1..5)
- loc, scale, minimum, maximum

Distribution semantics:
1 discrete: all mass at loc (vintage year, e.g., -12)
2 lognormal: defined on AGE (positive), ln(age) ~ Normal(loc, scale); vintage is -age
3 truncated normal: on VINTAGE directly, mean=loc, std=scale, truncated to [minimum, maximum]
4 uniform: on [minimum, maximum] (vintage)
5 triangular: min=minimum, mode=loc, max=maximum (vintage)

Notes:
- Type 2 uses "age" semantics for loc/scale, since lognormal does not support negative values.
  Downstream conversion: vintage = -age.
"""

from __future__ import annotations

import os
import json
import time
import hashlib
from typing import Any, Dict, Optional

import pandas as pd
from openai import OpenAI



In [2]:
INPUT_XLSX = "temporalized activities.xlsx"
OUTPUT_XLSX = "temporalized activities_filled.xlsx"
CACHE_JSON = "temporalized_cache.json"

MODEL = "gpt-5-mini"
SLEEP_S = 0.0
MAX_ROWS = 10

In [3]:

# -----------------------------
# Prompt policy (edit this)
# -----------------------------
INSTRUCTIONS = """
You parameterize (a) technical lifetime and (b) fleet vintage composition for durable goods/facilities.

Interpretation:
- year 0 = now.
- Vintage is commissioning/installation year relative to now (negative values are past, 0 is now).
- The vintage distribution should represent the CURRENT fleet composition (vintage mix of units in operation).

Return:
- lifetime: technical lifetime in years (>0)
- age_distribution_type: 1..5
- loc, scale, minimum, maximum: parameters per the rules below

Distribution types (allowed):
2 = lognormal on AGE (positive): ln(age) ~ Normal(loc, scale). Vintage = -age.
3 = truncated normal on VINTAGE: mean=loc, std=scale, truncated to [minimum, maximum].
5 = triangular on VINTAGE: on [minimum, maximum] with mode=loc.


General rules:
- Prefer vintage support <= 0 (do not place mass in the future unless strongly implied).
- Ensure minimum < maximum when provided.
- If the row does not correspond to a durable stock (e.g., energy carrier, consumable material, purely operational service),
  return null for all output fields.
- If uncertain, choose wider bounds and lower confidence.
- Use name, reference product, and any classification/taxonomy codes provided (ISIC/CPC/categories/etc.) to infer the most plausible lifetime and fleet vintage composition.
- Keep notes <= 12 words.
- Default choice: use type=5 unless there is a clear reason to use type=3 or type=2.


Hard consistency constraint (mandatory):
If the vintage distribution represents the current operating stock and the technical lifetime is L,
then the vintage support MUST cover (approximately) the full operating lifetime.

- Preferred default: type=5 (triangular on VINTAGE):
  set minimum = -L, loc = -L/2, maximum = -1.

- Alternative: type=3 (truncated normal on VINTAGE):
  set minimum = -L, maximum = -1, loc = -L/2, scale = L/6 (so most mass lies within the bounds).

- If using type=2 (lognormal on AGE):
  use AGE support concentrated between 1 and L years (no explicit min/max fields),
  and set loc/scale so the mean age is about L/2.

DO NOT use a narrow recent window (e.g., last 10 years) unless the row explicitly indicates
a newly introduced technology or a recent, system-wide replacement wave.


Engineering priors (typical):
- passenger car: 12–20 years
- heavy-duty truck: 10–18 years
- buildings: 40–80 years
- power plants: 30–80 years depending on tech (hydro often long-lived)
- industrial machinery: 10–25 years
- IT hardware: 3–8 years
"""

In [4]:

# -----------------------------
# Structured Outputs schema
# -----------------------------
BATCH_SCHEMA = {
    "name": "temporal_params_batch",
    "strict": True,
    "schema": {
        "type": "object",
        "additionalProperties": False,
        "properties": {
            "results": {
                "type": "array",
                "items": {
                    "type": "object",
                    "additionalProperties": False,
                    "properties": {
                        "row_id": {"type": "integer"},
                        "lifetime": {"type": ["number", "null"]},
                        "age_distribution_type": {"type": ["integer", "null"], "enum": [2, 3, 5, None]},
                        "loc": {"type": ["number", "null"]},
                        "scale": {"type": ["number", "null"]},
                        "minimum": {"type": ["number", "null"]},
                        "maximum": {"type": ["number", "null"]},
                        "confidence": {"type": "number", "minimum": 0.0, "maximum": 1.0},
                        "notes": {"type": "string"},
                    },
                    "required": [
                        "row_id", "lifetime", "age_distribution_type",
                        "loc", "scale", "minimum", "maximum", "confidence", "notes"
                    ],
                },
            }
        },
        "required": ["results"],
    },
}


def build_payload(row: pd.Series) -> Dict[str, Any]:
    payload = {
        "name": str(row.get("name", "")),
        "reference product": str(row.get("reference product", "")),
    }

    classification_like = [
        c for c in row.index
        if any(k in str(c).lower() for k in [
            "isic", "cpc", "category", "categories", "classification",
            "nace", "naics", "hs", "cn", "prodcom", "icb", "sic", "ecoinvent"
        ])
    ]
    for c in classification_like:
        v = row.get(c, None)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        payload[str(c)] = str(v)

    return payload


def _fingerprint(row: pd.Series) -> str:
    base = {
        "name": str(row.get("name", "")).strip(),
        "reference product": str(row.get("reference product", "")).strip(),
    }

    classification_like = [
        c for c in row.index
        if any(k in str(c).lower() for k in [
            "isic", "cpc", "category", "categories", "classification",
            "nace", "naics", "hs", "cn", "prodcom", "icb", "sic", "ecoinvent"
        ])
    ]
    for c in classification_like:
        v = row.get(c, None)
        if v is None or (isinstance(v, float) and pd.isna(v)):
            continue
        base[str(c)] = str(v).strip()

    return hashlib.sha256(json.dumps(base, sort_keys=True).encode("utf-8")).hexdigest()


def _as_number(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return None

def _extract_json_text(resp) -> str:
    """
    Try to get the JSON text from a Responses API response across SDK variants.
    Raises a helpful error if no text is present.
    """
    # Preferred helper if present
    txt = getattr(resp, "output_text", None)
    if isinstance(txt, str) and txt.strip():
        return txt.strip()

    # Fall back to scanning output blocks
    out = getattr(resp, "output", None)
    if isinstance(out, list):
        chunks = []
        for item in out:
            # item is typically a dict-like object
            content = item.get("content") if isinstance(item, dict) else getattr(item, "content", None)
            if isinstance(content, list):
                for c in content:
                    ctype = c.get("type") if isinstance(c, dict) else getattr(c, "type", None)
                    if ctype == "output_text":
                        text = c.get("text") if isinstance(c, dict) else getattr(c, "text", None)
                        if isinstance(text, str) and text.strip():
                            chunks.append(text.strip())
        if chunks:
            return "\n".join(chunks)

    raise ValueError(
        "No output_text found in response. "
        "Print `resp` (or `resp.model_dump()` if available) to inspect the returned structure."
    )


def validate_and_repair(out: Dict[str, Any]) -> Dict[str, Any]:
    """
    Enforce basic consistency. If inconsistent, repair conservatively.
    """
    t = out.get("age_distribution_type", None)

    # Restrict allowed distribution types
    if t not in (2, 3, 5, None):
        t = None
        out["age_distribution_type"] = None
        
    lifetime = _as_number(out.get("lifetime"))
    loc = _as_number(out.get("loc"))
    scale = _as_number(out.get("scale"))
    mn = _as_number(out.get("minimum"))
    mx = _as_number(out.get("maximum"))

    # If model returned "nulls", pass through
    if lifetime is None and t is None and loc is None and scale is None and mn is None and mx is None:
        return {**out, "lifetime": None, "age_distribution_type": None, "loc": None, "scale": None, "minimum": None, "maximum": None}

    # Lifetime must be > 0
    if lifetime is None or lifetime <= 0:
        # If we can infer from bounds for uniform/triangular, do so; otherwise null it out.
        if t == 4 and mn is not None and mx is not None and mn < mx:
            # mean age = -mean(vintage) if vintage is negative
            # but lifetime is technical lifetime; set to abs(midpoint) if plausible fallback
            lifetime = max(1.0, abs((mn + mx) / 2.0))
        elif t == 5 and mn is not None and mx is not None and loc is not None and mn < mx:
            lifetime = max(1.0, abs((mn + mx + loc) / 3.0))
        else:
            lifetime = None

    # Type-specific checks
    if t == 1:
        # discrete: need loc, vintage typically <=0
        if loc is None:
            loc = -lifetime if lifetime else None
        # optional: clip to <=0
        if loc is not None and loc > 0:
            loc = 0.0
        scale = None
        mn = None
        mx = None

    elif t == 2:
        # lognormal on AGE: need loc & scale, scale>0; loc can be any real
        if scale is None or scale <= 0:
            scale = 0.4  # conservative
        if loc is None:
            # approximate from lifetime if present: mean age = exp(mu + 0.5*sigma^2)
            # => mu = ln(mean) - 0.5*sigma^2
            if lifetime and lifetime > 0:
                import math
                loc = math.log(lifetime) - 0.5 * (scale ** 2)
            else:
                loc = 2.0
        mn = None
        mx = None

    elif t == 3:
        # truncated normal on VINTAGE: need loc & scale & bounds
        if mn is None or mx is None:
            # Default to a 30-year span ending recently
            mx = -1.0
            mn = -31.0
        if mn >= mx:
            mn, mx = min(mn, mx) - 1.0, max(mn, mx) + 1.0
        if loc is None:
            loc = (mn + mx) / 2.0
        if scale is None or scale <= 0:
            scale = max(1.0, abs(mx - mn) / 6.0)  # ~99% within bounds for normal
        # keep support <=0 if plausible
        if mx > 0:
            mx = 0.0
            if mn >= mx:
                mn = mx - 1.0

    elif t == 4:
        # uniform: need bounds
        if mn is None or mx is None:
            mx = -1.0
            mn = -11.0
        if mn >= mx:
            mn, mx = min(mn, mx) - 1.0, max(mn, mx) + 1.0
        # prefer not future
        if mx > 0:
            mx = 0.0
            if mn >= mx:
                mn = mx - 1.0
        loc = None
        scale = None

    elif t == 5:
        # triangular: need mn, mx, loc(mode)
        if mn is None or mx is None:
            mx = -1.0
            mn = -21.0
        if mn >= mx:
            mn, mx = min(mn, mx) - 1.0, max(mn, mx) + 1.0
        if loc is None:
            loc = (mn + mx) / 2.0
        # clip mode into [mn, mx]
        if loc < mn:
            loc = mn
        if loc > mx:
            loc = mx
        if mx > 0:
            mx = 0.0
            if mn >= mx:
                mn = mx - 1.0
            if loc > mx:
                loc = mx
        scale = None

    else:
        # Unknown => null out
        return {**out, "lifetime": None, "age_distribution_type": None, "loc": None, "scale": None, "minimum": None, "maximum": None}

    out["lifetime"] = lifetime
    out["age_distribution_type"] = t
    out["loc"] = loc
    out["scale"] = scale
    out["minimum"] = mn
    out["maximum"] = mx
    return out


def query_model_batch(client: OpenAI, batch_items: list[dict]) -> Dict[int, Dict[str, Any]]:
    """
    batch_items: list of {"row_id": int, "payload": dict}
    returns: mapping row_id -> output dict
    """
    # Keep the prompt small and regular
    input_obj = {
        "rows": [
            {"row_id": item["row_id"], "row": item["payload"]}
            for item in batch_items
        ]
    }

    resp = client.responses.create(
        model=MODEL,
        instructions=INSTRUCTIONS,
        input=json.dumps(input_obj, ensure_ascii=False),
        #temperature=0,
        reasoning={"effort": "low"},
        max_output_tokens=3000,  # adjust based on batch size
        text={
            "format": {
                "type": "json_schema",
                "name": BATCH_SCHEMA["name"],
                "schema": BATCH_SCHEMA["schema"],
                "strict": BATCH_SCHEMA["strict"],
            }
        },
    )

    try:
        raw = _extract_json_text(resp)
        parsed = json.loads(raw)
    except Exception as e:
        # Helpful debugging info
        print("Failed to parse JSON from response.")
        print("resp.output_text repr:", repr(getattr(resp, "output_text", None)))
        # If available, dump the whole response
        if hasattr(resp, "model_dump"):
            print("Full response:", json.dumps(resp.model_dump(), ensure_ascii=False)[:4000])
        raise
    
    outputs = parsed["results"]
    return {int(o["row_id"]): o for o in outputs}

In [5]:
client = OpenAI()
df = pd.read_excel(INPUT_XLSX)

# Load cache
cache: Dict[str, Dict[str, Any]] = {}
if os.path.exists(CACHE_JSON):
    with open(CACHE_JSON, "r", encoding="utf-8") as f:
        cache = json.load(f)

BATCH_SIZE = 20

pending = [] 
n_rows = len(df)

for i in range(n_rows):
    print(i)
    row = df.iloc[i]

    if pd.notna(row.get("lifetime")) and pd.notna(row.get("age distribution type")):
        continue

    fp = _fingerprint(row)

    if fp in cache:
        out = cache[fp]
        # write immediately
        df.at[i, "lifetime"] = out.get("lifetime")
        df.at[i, "age distribution type"] = out.get("age_distribution_type")
        df.at[i, "loc"] = out.get("loc")
        df.at[i, "scale"] = out.get("scale")
        df.at[i, "minimum"] = out.get("minimum")
        df.at[i, "maximum"] = out.get("maximum")
        continue

    pending.append({"row_id": i, "payload": build_payload(row), "fp": fp})

    if len(pending) >= BATCH_SIZE:
        batch_map = query_model_batch(client, pending)
        for item in pending:
            rid = item["row_id"]
            fp = item["fp"]
        
            if rid not in batch_map:
                # Skip gracefully; optionally handle with a retry strategy later
                # (e.g., collect missing rows and re-run them in a smaller batch)
                continue
        
            out = validate_and_repair(batch_map[rid])
            cache[fp] = out

            df.at[rid, "lifetime"] = out.get("lifetime")
            df.at[rid, "age distribution type"] = out.get("age_distribution_type")
            df.at[rid, "loc"] = out.get("loc")
            df.at[rid, "scale"] = out.get("scale")
            df.at[rid, "minimum"] = out.get("minimum")
            df.at[rid, "maximum"] = out.get("maximum")

        with open(CACHE_JSON, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)

        df.to_excel(OUTPUT_XLSX, index=False)
        
        pending = []
        # Remove sleep for batching; if needed, keep a tiny one
        # time.sleep(SLEEP_S)

# flush remainder
if pending:
    batch_map = query_model_batch(client, pending)
    for item in pending:
        rid = item["row_id"]
        fp = item["fp"]
    
        if rid not in batch_map:
            # Skip gracefully; optionally handle with a retry strategy later
            # (e.g., collect missing rows and re-run them in a smaller batch)
            continue
    
        out = validate_and_repair(batch_map[rid])
        cache[fp] = out

        df.at[rid, "lifetime"] = out.get("lifetime")
        df.at[rid, "age distribution type"] = out.get("age_distribution_type")
        df.at[rid, "loc"] = out.get("loc")
        df.at[rid, "scale"] = out.get("scale")
        df.at[rid, "minimum"] = out.get("minimum")
        df.at[rid, "maximum"] = out.get("maximum")

    with open(CACHE_JSON, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)


print(f"Wrote {OUTPUT_XLSX}")
print(f"Cache: {CACHE_JSON} (size={len(cache)})")

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27